In [1]:
import pandas as pd

In [2]:
def oracle(df, opt):
    print('Energy overhead of the dynamic approach compared to an oracle-based approach')
    print('-x% means that we consume x% less energy than the oracle, x% means that we consume x% more energy')

    mt = df[df['threads'] == 'mt'].copy()
    df = df[(df['threads'] != 'mt') & (df['threads'] != 'rt')].copy()

    mt['runtime'] = mt.apply(lambda row: row['runtime'] / df[(df['size'] == row['size']) & (df['threads'] == opt[row['size']])].iloc[0]['runtime'] * 100 - 100, axis=1)
    mt['energy']  = mt.apply(lambda row: row['energy']  / df[(df['size'] == row['size']) & (df['threads'] == opt[row['size']])].iloc[0]['energy']  * 100 - 100, axis=1)

    return mt.groupby('size').mean(numeric_only=True).round(0).astype(int)

In [3]:
def runtime(df):
    print('Energy overhead of the dynamic approach compared to a runtime-based approach')
    print('-x% means that we consume x% less energy than the rt-based approach, x% means that we consume x% more energy')

    mt = df[df['threads'] == 'mt'].copy()
    rt = df[df['threads'] == 'rt'].copy()

    mt['runtime'] = mt.apply(lambda row: row['runtime'] / rt[rt['size'] == row['size']].iloc[0]['runtime'] * 100 - 100, axis=1)
    mt['energy']  = mt.apply(lambda row: row['energy']  / rt[rt['size'] == row['size']].iloc[0]['energy']  * 100 - 100, axis=1)

    return mt.groupby('size').mean(numeric_only=True).round(0).astype(int)

In [4]:
def mean(df):
    print('\nAverage energy overhead of the dynamic approach compared to a static approach')
    print('x% means that, averaged over all input sizes, we are x% more energy efficient')
    
    mt = df[df['threads'] == 'mt'].copy()
    df = df[(df['threads'] != 'mt') & (df['threads'] != 'rt')].copy()

    # Normalise runtime and energy consumption
    df['runtime'] = df.apply(lambda row: 100 - mt[mt['size'] == row['size']].iloc[0]['runtime'] / row['runtime'] * 100, axis=1)
    df['energy']  = df.apply(lambda row: 100 - mt[mt['size'] == row['size']].iloc[0]['energy']  / row['energy']  * 100, axis=1)

    # Drop unused columns and rows
    df = df.drop('size', axis=1)
    df = df[df['threads'] != 'mt']
    df['threads'] = df['threads'].astype(int)

    # Mean per thread-count
    return df.groupby('threads').mean(numeric_only=True).round(0).astype(int)

In [13]:
df = pd.read_csv('data/compare_matmul.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)

print(oracle(df, { 500: '16', 1000: '16', 1500: '1' }))
print(runtime(df))
print(mean(df))

Energy overhead of the dynamic approach compared to an oracle-based approach
-x% means that we consume x% less energy than the oracle, x% means that we consume x% more energy
      runtime  energy
size                 
500        17       9
1000       13      10
1500      -25      75
Energy overhead of the dynamic approach compared to a runtime-based approach
-x% means that we consume x% less energy than the rt-based approach, x% means that we consume x% more energy
      runtime  energy
size                 
500         3       2
1000       -1       4
1500        1       1

Average energy overhead of the dynamic approach compared to a static approach
x% means that, averaged over all input sizes, we are x% more energy efficient
         runtime  energy
threads                 
1             55      -8
8             41      12
12            15       2
16           -12      -7


In [10]:
df = pd.read_csv('data/compare_nbody.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)

print(oracle(df, { 10000: '16', 25000: '16', 40000: '16' }))
print(runtime(df))
print(mean(df))

Energy overhead of the dynamic approach compared to an oracle-based approach
-x% means that we consume x% less energy than the oracle, x% means that we consume x% more energy
       runtime  energy
size                  
10000        7       3
25000        6      -3
40000       23      -1
Energy overhead of the dynamic approach compared to a runtime-based approach
-x% means that we consume x% less energy than the rt-based approach, x% means that we consume x% more energy
       runtime  energy
size                  
10000        1       1
25000       -2      -2
40000        6       2

Average energy overhead of the dynamic approach compared to a static approach
x% means that, averaged over all input sizes, we are x% more energy efficient
         runtime  energy
threads                 
8             36       9
12             6      -2
16           -12       0


In [12]:
df = pd.read_csv('data/compare_stencil.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)

print(oracle(df, { 10000: '16', 25000: '14', 40000: '12' }))
print(runtime(df))
print(mean(df))

Energy overhead of the dynamic approach compared to an oracle-based approach
-x% means that we consume x% less energy than the oracle, x% means that we consume x% more energy
       runtime  energy
size                  
10000        6       5
25000        6       5
40000       30      14
Energy overhead of the dynamic approach compared to a runtime-based approach
-x% means that we consume x% less energy than the rt-based approach, x% means that we consume x% more energy
       runtime  energy
size                  
10000       -9      -3
25000       -4      -5
40000       -3      -2

Average energy overhead of the dynamic approach compared to a static approach
x% means that, averaged over all input sizes, we are x% more energy efficient
         runtime  energy
threads                 
8             33      11
12             0      -3
14            -7      -2
16            -7       4


In [8]:
df = pd.read_csv('data/compare_rust.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)

y = df[df['pin'] == True].copy()
print(oracle(y, { 500: '16', 1000: '16', 1500: '12' }))
print(runtime(y))
print(mean(y))

Energy overhead of the dynamic approach compared to an oracle-based approach
-x% means that we consume x% less energy than the oracle, x% means that we consume x% more energy
      pin  runtime  energy
size                      
500     1        6       2
1000    1        5       2
1500    1        8       6
Energy overhead of the dynamic approach compared to a runtime-based approach
-x% means that we consume x% less energy than the rt-based approach, x% means that we consume x% more energy
      pin  runtime  energy
size                      
500     1       -6      -3
1000    1       -3      -1
1500    1      -10      -5

Average energy overhead of the dynamic approach compared to a static approach
x% means that, averaged over all input sizes, we are x% more energy efficient
         pin  runtime  energy
threads                      
8          1       50      23
12         1       -3      -2
16         1        1       3


In [9]:
df = pd.read_csv('data/compare_rust.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)

n = df[df['pin'] == False].copy()
print(oracle(n, { 500: '16', 1000: '16', 1500: '8' }))
print(runtime(n))
print(mean(n))

Energy overhead of the dynamic approach compared to an oracle-based approach
-x% means that we consume x% less energy than the oracle, x% means that we consume x% more energy
      pin  runtime  energy
size                      
500     0        2       0
1000    0        1       1
1500    0       27      13
Energy overhead of the dynamic approach compared to a runtime-based approach
-x% means that we consume x% less energy than the rt-based approach, x% means that we consume x% more energy
      pin  runtime  energy
size                      
500     0       -3      -2
1000    0       -1      -1
1500    0       -1       2

Average energy overhead of the dynamic approach compared to a static approach
x% means that, averaged over all input sizes, we are x% more energy efficient
         pin  runtime  energy
threads                      
8          0        2       0
12         0        1       2
16         0        4       6
